In [15]:
!pip install pymupdf 
!pip install contractions 
!pip install num2words
import num2words as n2w
import contractions as ctr
import pandas as pd
import pymupdf
import re
import string


   ---------- ----------------------------- 1/4 [anyascii]
   ---------------------------------------- 4/4 [contractions]

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13856 sha256=08f41031e6e6d41c10ccec2c5a110ff035942cecf76b5c662f70342ec691286f
  Stored in directory: c:\users\khuong nguyen\appdata\local\pip\cache\wheels\0b\1d\03\175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt

   ---------------------------------------- 2/2 [num2words]



# 1. Extract Data from reference progress note + Get Data from NurseGPT (assumed CSV file)


In [ ]:
# Get refernce text:
mapping_color_doc = pd.read_excel(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\Simulate_Tier_Mapping_For_ProgressNote.xlsx")
ref_notes = pymupdf.open(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\Progress notes.pdf")
hypo_notes = pd.read_csv(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\hypo_notes.csv") #Assume NurseGPT's artifacts stored as csv/ xlsx

def reg_pattern_find(text, page_number):
    pattern_list = {
         "Reference Diagnoses": r"Diagnoses\s*:(.*?)(?=\n[A-Z][a-z]+\s*:|$)",
    }
    result = {}
    
    for field_name, pattern in pattern_list.items():
        match = re.search(pattern, text, re.DOTALL)
        if match:
            result[field_name] =  match.group(1) 
        else:
             result[field_name] = None
             print(f"Page {page_number+1} does not contain the pattern, please recheck this page content")
    return result 
             

# Extract content from PDFs -> Convert into DF

def note_extraction(note_doc, mapping_doc):
    rows = []
    for page_num, page in enumerate(note_doc, start = 1): #Loop through each progress note (1 note = 1 page)
        note_content = page.get_text() #Get ALL content fields (Diagnoses details, Date Created, Physicians ID,etc)
        tier = mapping_color_doc.loc[mapping_color_doc['Page Number'] == page_num, 'Tier'].values[0]
        
        field_results = reg_pattern_find(note_content, page_num)
        field_results["Page Number"] = page_num
        field_results['Color Tier'] = tier 
        rows.append(field_results)
        
        reference_note_df = pd.DataFrame(rows)
        
    return reference_note_df 


#Set DF display option
pd.set_option('display.max_colwidth', None)
ref_notes = note_extraction(ref_notes, mapping_color_doc)
ref_notes

,Reference Diagnoses,Page Number,Color Tier
0,"\nMagnesium deficiency(E61.2), Abnormal findings on diagnostic imaging of heart and coronary circulation(R93.1), Presence of\naortocoronary bypass graft(Z95.1), Type 2 diabetes mellitus with other specified kidney complication not elsewhere classified(E11.28),\nDiverticular disease of intestine, part unspecified, without perforation or abscess(K57.9), Other degenerative disorders of globe(H44.3),\nAbnormal cardiovascular function studies (biomarkers or ECG) suggestive of non ST segment elevation myocardial infarction [NSTEMI]\n(R94.31), Chronic kidney disease, unspecified(N18.9), Benign hypertension(I10.0), Epistaxis(R04.0), Stroke, not specified as haemorrhage\nor infarction(I64), Vascular dementia, unspecified(F01.9), Atrial fibrillation, unspecified(I48.90), Sleep apnoea, obstructed(G47.30),\nCongestive heart failure(I50.0), Vitamin B12 deficiency anaemia, unspecified(D51.9), Cardiovascular disease, unspecified(I51.6), Personal\nhistory of COVID-19(U07.5), Dysphasia and aphasia(R47.0)\nCreated Date: 04/13/2026 08:16\nEffective Date: 04/13/2026 08:15",1,Green


# 2. Data Preprocessing Steps

In [32]:
spoken_ref_note = pd.read_csv(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\nurse_spoken_script(Claude).csv", index_col = False)
nurse_gpt_note = pd.read_csv(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\nursegpt_output_script(Claude).csv", index_col = False)

In [33]:
spoken_ref_note

,Reference Diagnoses,Page Number,Color Tier
0,"At approximately seven o'clock PM, I responded to an emergency call bell near the third floor elevator. Upon arrival, I found Mrs. Jane Doe, that's room number one twenty-one, eighty-two years old, lying on her back with a care aide supporting her head.\nThe care aide witnessed the fall and reported that the resident was reaching for the elevator button when she lost her balance and fell backward, striking the back of her head on the tile floor. Mrs. Doe was alert but confused about what happened.\nThere was a three centimeter hematoma on the back of her head, but no open wounds. Her Glasgow Coma Scale was thirteen - eye response four, verbal response four, and motor response five.\nI checked her vitals. Blood pressure was one sixty over ninety, heart rate eighty-eight, respiratory rate eighteen, and oxygen saturation ninety-six percent on room air.\nI initiated neuro checks every fifteen minutes and gave her six hundred fifty milligrams of Tylenol per standing orders. I paged the on-call physician, but didn't get an immediate response.\nSince this was a witnessed fall with a head impact, I called nine-one-one for hospital transfer. I also notified Mrs. Doe's power of attorney - that's her husband, George.\nThe resident was transferred to the hospital at seven thirty-five PM, along with her med administration record, recent vitals, and her advance directives. I've let facility maintenance know to check the lighting in that elevator area. I'll follow up with the hospital for updates.",1,Green


In [34]:
nurse_gpt_note

,Hypothesis Diagnoses,Page Number,Color Tier
0,"At approximately 19:00, I responded to an emergency call bell near the third floor elevator. Upon arrival, I found Mrs. Jane Doe (Room Number-121), 82 years old, lying on her back with a care aide supporting her head. The care aide witnessed the fall and reported that the resident was reaching for the elevator button when she lost her balance and fell backward, striking the back of her head on the tile floor. Mrs. Doe was alert but confused about what happened. There was a three-centimeter hematoma on the back of her head but no open wounds. Her Glasgow Coma Scale was thirteen, with eye response at four, verbal response at four, and motor response at five. I checked her vitals: blood pressure was one sixty over ninety, heart rate was eighty-eight, respiratory rate was eighteen, and oxygen saturation was ninety-six percent on room air. I initiated neurological checks every fifteen minutes and administered six hundred fifty milligrams of Tylenol as per standing orders. I paged the on-call physician, but there was no immediate response. Given that this was a witnessed fall with a head impact, I called nine-one-one for hospital transfer. Mrs. Doe's power of attorney, her husband George, was notified. The resident was transferred to the hospital at nineteen thirty-five with her medication administration record, recent vitals, and advance directives. Facility maintenance has been informed to check the lighting in the elevator area. Will follow up with the hospital for updates",1,Green


In [ ]:
spoken_ref_note = pd.read_csv("")
def contraction(text):
    if isinstance(text, str):
        return ctr.fix(text)
    else:
        return text

def convert_number_words(match):
    number_str = match.group(0)
    number_int = int(number_str)
    return n2w.num2words(number_int)



def preprocessing_pipeline(df, column):
    df[column] = df[column].str.lower()  # Decapitalization
    df[column] = df[column].str.replace('\n', ' ')  # Remove newline

    df[column] = df[column].apply(lambda x: re.sub(r'\d+', convert_number_words, x))  # Convert numbers to words (82 -> 'eighty-two')

    df[column] = df[column].str.replace('-', ' ')
    df[column] = df[column].str.replace(':', ' ')
    df[column] = df[column].str.replace('/', 'over')
    df[column] = df[column].str.replace(r"'s\b", '', regex=True)
    # Contraction
    df[column] = df[column].apply(contraction)

    # Removal of punctuation
    punc_table = str.maketrans("", "", string.punctuation)
    df[column] = df[column].str.translate(punc_table)

    # Removal of white space
    df[column] = df[column].str.replace(r'\s+', ' ', regex=True)  # Extra white space
    df[column] = df[column].str.strip()  # Trailing white space

    return df

preprocessing_pipeline(spoken_ref_note, 'Reference Diagnoses')

FileNotFoundError: [Errno 2] No such file or directory: ''